In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 5.13 Random Walks: From Coin Flips to the Diffusion Equation

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume V — Classical Statistical Mechanics",
    number="5.13",
    title="Random Walks: From Coin Flips to the Diffusion Equation",
    blurb="The humblest stochastic process in physics, taken seriously: "
    "coin flips growing a Gaussian, a walker's histogram solving a partial "
    "differential equation, first passages that always happen but take "
    "forever on average, Pólya's drunkard finding home in two dimensions "
    "and losing it in three, and a polymer that swells because it may not "
    "cross itself.",
    difficulty="intermediate",
    estimate="130–160 min",
)

## Notebook overview

In 1905 Karl Pearson asked the readers of *Nature* for help with what he
called the problem of the random walk — a man walks a fixed stride in a
random direction, repeats, where is he? — and Rayleigh answered within
weeks, because he had solved it decades earlier for sound waves
{cite}`pearson1905`. That exchange is the founding joke of the subject:
the random walk is *everywhere*, worked out by someone else, for
something else. It is Brownian motion coarse-grained, diffusion
discretised, a polymer's backbone, a gambler's bankroll, and the
hidden solver inside half of computational physics.

This notebook gives the walk the systematic treatment the course has so
far scattered: [§0.11](../00-foundations/random-numbers-monte-carlo.ipynb)
built the random numbers, [§5.3](large-n-limit.ipynb) proved the central
limit theorem that governs the walk's envelope, and
[§5.11](taste-of-nonequilibrium.ipynb) met Brownian motion from the
Langevin side. Here we work the *lattice* side: the exact binomial, its
Gaussian limit, and the passage to the diffusion *equation* — then the
questions only the walk itself can answer. When does a walker first
reach a boundary (the answer's $t^{-3/2}$ tail means "certainly, but on
average never")? Does it return home (Pólya's theorem: yes in one and
two dimensions, only every third time in three {cite}`polya1921`)? And
what changes when the walk may not cross itself (the polymer question:
it *swells*, with a universal exponent)? The classic survey of all of it
is Chandrasekhar {cite}`chandrasekhar1943`; the volume's standing
reference remains {cite}`nolting6`.

A note on reading the checks in this notebook: a validation compares a
result to an expected physical fact. A ✗ does not by itself mean the
answer is wrong; it means the output did not match what the check
expected, which may be a genuine error, a different-but-valid
convention, or too tight a tolerance. Treat a ✗ as a prompt to locate
the discrepancy. Passing is strong evidence, not proof.

## Theory in brief

**The simple walk and its two laws.** A walker on the integers takes
independent steps $\pm 1$ with equal probability. After $N$ steps the
displacement $x_N = \sum_k s_k$ obeys the *exact* binomial law

```{math}
:label: eq-rw-binomial
P(x_N = x) \;=\; \binom{N}{(N+x)/2} 2^{-N}
\qquad (x \equiv N \bmod 2),
```

and two moments carry the physics: $\langle x_N\rangle = 0$ and
$\langle x_N^2\rangle = N$ — the root-mean-square excursion grows as
$\sqrt N$, the single most consequential square root in statistical
physics. For large $N$ the central limit theorem of
[§5.3](large-n-limit.ipynb) flattens the binomial into the Gaussian of
variance $N$.

**From walk to PDE.** Let the walker step every $\tau$ on a lattice of
spacing $a$, and write $P(x, t)$ for the occupation probability. The
master equation $P(x, t+\tau) = \tfrac12 P(x-a, t) + \tfrac12 P(x+a,t)$
Taylor-expands, for slow spatial variation, into

```{math}
:label: eq-rw-diffusion
\frac{\partial P}{\partial t} \;=\; D\,
\frac{\partial^2 P}{\partial x^2},
\qquad D \;=\; \frac{a^2}{2\tau},
```

the diffusion equation: the walk *is* the PDE, coarse-grained, and the
correspondence is quantitative — a walker histogram and a finite-
difference solution of {eq}`eq-rw-diffusion` must agree bin by bin.
(The FTCS scheme that solves it is stable for $D\,\Delta t/\Delta x^2
\le 1/2$; the walk itself sits exactly *at* the stability boundary,
which is a nice way to remember the criterion.)

**First passage.** Start at $0$ and ask when the walker first reaches
$+1$. The reflection principle gives the classic answer: the
first-passage probability decays as

```{math}
:label: eq-rw-firstpassage
F(t) \;\sim\; \frac{1}{\sqrt{2\pi}}\, t^{-3/2}
\qquad (t \to \infty),
```

whose sum converges ($ \sum F = 1$: passage is *certain*) while its
first moment diverges ($\sum t F = \infty$: the *mean* wait is
infinite). Heavy tails are not pathology; they are the generic
geometry of unbiased wandering, and the same $-3/2$ governs everything
from bond returns to neuron firing models.

**Absorbing boundaries: the gambler's ruin.** Between two absorbing
walls at $-a$ and $+b$ the martingale (fair-game) property fixes the
exact absorption probabilities and the mean game length with no
asymptotics at all:

```{math}
:label: eq-rw-ruin
P(\text{hit } -a \text{ first}) \;=\; \frac{b}{a+b},
\qquad
\langle T \rangle \;=\; a\,b .
```

**Recurrence: Pólya's theorem.** Does the walker return to its
starting point? Pólya {cite}`polya1921` proved the answer depends only
on dimension: return is *certain* in one and two dimensions and
uncertain in three, where the return probability is the Watson-integral
value $p_3 = 1 - 1/u_3 = 0.340\,537\ldots$, evaluated in closed form by
Glasser and Zucker {cite}`glasser1977` as a product of Gamma functions:

```{math}
:label: eq-rw-polya
u_3 \;=\; \frac{\sqrt 6}{32\pi^3}\,
\Gamma\!\left(\tfrac{1}{24}\right)
\Gamma\!\left(\tfrac{5}{24}\right)
\Gamma\!\left(\tfrac{7}{24}\right)
\Gamma\!\left(\tfrac{11}{24}\right).
```

A drunk man will find his way home; a drunk bird may be lost forever.

**The self-avoiding walk.** Forbid the walker from revisiting any site
— the minimal model of a polymer whose monomers cannot overlap — and
the $\sqrt N$ law fails *upward*: $\langle R_N^2\rangle \sim N^{2\nu}$
with $\nu = 3/4$ exactly in two dimensions (Flory's estimate, later
proved exact), against the simple walk's $\nu = 1/2$. Excluded volume
is not a correction; it changes the exponent, and exponents are what
[§5.10](ising-emergence-universality.ipynb) taught us to treasure.
Sampling self-avoiding walks fairly requires care: naive generation
dies young (most walks trap themselves), and Rosenbluth's fix — grow
step by step among the *open* neighbours, carrying a compensating
weight equal to the product of open-neighbour counts — is one of the
founding algorithms of polymer physics and a direct ancestor of the
importance-sampling ideas of
[§0.11](../00-foundations/random-numbers-monte-carlo.ipynb).

## Setup

Dimensionless lattice units throughout: step length $1$, step time $1$,
so $D = 1/2$ by {eq}`eq-rw-diffusion`. All randomness from the seeded
generator.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from scipy.special import comb, gamma

from ecp import animate, validate

rng = np.random.default_rng(0)


def walk_ensemble(n_walkers, n_steps, rng):
    """Final positions of an ensemble of simple ±1 walks.

    Vectorized: draws the full (n_walkers, n_steps) sign matrix and sums
    along steps — the whole ensemble advances as one array operation.

    Parameters
    ----------
    n_walkers : int
        Number of independent walkers.
    n_steps : int
        Steps per walker.
    rng : numpy.random.Generator
        Seeded generator.

    Returns
    -------
    numpy.ndarray
        Final displacements, shape (n_walkers,).
    """
    steps = rng.choice([-1, 1], size=(n_walkers, n_steps))
    return steps.sum(axis=1)


def polya_p3_exact():
    """Pólya's 3D return probability from the Glasser–Zucker closed form.

    Evaluates eq-rw-polya with scipy.special.gamma and returns
    1 - 1/u_3 = 0.340537...: the exact probability that a simple cubic
    random walk ever returns to its origin.

    Returns
    -------
    float
        The return probability p_3.
    """
    u3 = (
        np.sqrt(6.0)
        / (32.0 * np.pi**3)
        * gamma(1.0 / 24.0)
        * gamma(5.0 / 24.0)
        * gamma(7.0 / 24.0)
        * gamma(11.0 / 24.0)
    )
    return 1.0 - 1.0 / u3


def rosenbluth_r2(n_steps, n_samples, rng):
    """Rosenbluth-weighted mean squared end-to-end distance of 2D SAWs.

    Grows each self-avoiding walk step by step among the currently open
    lattice neighbours, accumulating the Rosenbluth weight (the product
    of open-neighbour counts) that exactly compensates the growth bias;
    trapped walks contribute nothing. The weighted average
    sum(w R^2)/sum(w) is an unbiased estimator of <R^2> over uniform
    self-avoiding walks.

    Parameters
    ----------
    n_steps : int
        Walk length.
    n_samples : int
        Number of attempted walks.
    rng : numpy.random.Generator
        Seeded generator.

    Returns
    -------
    float
        The weighted mean squared end-to-end distance.
    """
    moves = ((1, 0), (-1, 0), (0, 1), (0, -1))
    r2_weighted = 0.0
    weight_sum = 0.0
    for _ in range(n_samples):
        occupied = {(0, 0)}
        x, y = 0, 0
        w = 1.0
        alive = True
        for _ in range(n_steps):
            open_moves = [
                (dx, dy) for dx, dy in moves if (x + dx, y + dy) not in occupied
            ]
            if not open_moves:
                alive = False
                break
            w *= len(open_moves)
            dx, dy = open_moves[rng.integers(0, len(open_moves))]
            x, y = x + dx, y + dy
            occupied.add((x, y))
        if alive:
            r2_weighted += w * (x * x + y * y)
            weight_sum += w
    return r2_weighted / weight_sum

## Exercise 1 — The exact walk and its Gaussian shadow

{eq}`eq-rw-binomial` is exact at every $N$, so the first checks are
against arithmetic, not asymptotics.

**Part a)** Simulate $2\times10^5$ walkers for $N = 40$ steps with
`walk_ensemble` and verify: (i) $\langle x^2\rangle / N = 1$ within $4$
standard errors of the mean (the variance of $x^2$ for a sum of signs
is $2N(N-1)$, so the standard error is $\sqrt{2N(N-1)/M}/N$ for $M$
walkers); (ii) the full histogram matches the exact binomial
{eq}`eq-rw-binomial` evaluated by `scipy.special.comb`: every even bin
within $5$ Poisson standard deviations, and the total variation
distance $\tfrac12\sum_x |\hat P(x) - P(x)|$ below $0.01$.

**Part b)** Watch the Gaussian arrive. For $N = 4, 16, 64, 256$
overlay the exact binomial (rescaled by $\sqrt N$) on the standard
Gaussian: verify the maximum deviation between the rescaled binomial
and the Gaussian density falls monotonically with $N$ and drops below
$2\%$ at $N = 256$ — the central limit theorem of
[§5.3](large-n-limit.ipynb), watched frame by frame.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    abs(x2_over_N - 1.0) < 4.0 * se_x2,
    "the second moment obeys <x²> = N within 4σ: the √N law, the most "
    "consequential square root in statistical physics",
    f"<x²>/N = {x2_over_N:.4f} ± {se_x2:.4f}",
)
validate.check(
    tv_dist < 0.01 and max_bin_dev < 5.0,
    "the sampled histogram matches the exact binomial bin by bin "
    "(worst bin within 5 Poisson σ, total variation < 1%)",
    f"TV = {tv_dist:.4f}, worst {max_bin_dev:.1f}σ",
)
validate.check(
    all(a > b for a, b in zip(gauss_dev, gauss_dev[1:])) and gauss_dev[-1] < 0.02,
    "and the rescaled binomial marches monotonically onto the Gaussian, "
    "landing within 2% by N = 256: the CLT, frame by frame",
    f"deviations {['%.3f' % d for d in gauss_dev]}",
)

## Exercise 2 — The walker histogram solves a PDE

{eq}`eq-rw-diffusion` claims the walk and the diffusion equation are
the same object at two magnifications, and the claim is checkable to
histogram accuracy. With step length and time both $1$, the walk's
diffusion constant is $D = 1/2$.

**Part a)** Evolve $2\times10^5$ walkers for $T = 400$ steps. Solve
{eq}`eq-rw-diffusion` with the FTCS scheme — `P[1:-1] += r*(P[2:] -
2*P[1:-1] + P[:-2])` with $r = D\,\Delta t/\Delta x^2$ on the grid
$x \in [-120, 120]$, $\Delta x = 1$, $\Delta t = 0.2$ (so $r = 0.1$,
comfortably inside the stability bound $1/2$) — from a delta function
at the origin to the same time. Verify the two agree: pool the
walker histogram and the FTCS solution into parity-summed bins (the
lattice walk lives on even sites at even times; summing adjacent site
pairs removes the parity comb) and require the maximum absolute
difference below $2\times10^{-3}$ against a peak near
$2\times10^{-2}$.

**Part b)** Verify the moment law both ways: the walkers'
$\langle x^2\rangle$ at $T = 100, 200, 400$ matches $2Dt = t$ within
$4$ standard errors each, and the FTCS solution's discrete second
moment $\sum x^2 P \Delta x$ at $T = 400$ matches $400$ to
`rtol=1e-3`: one law, measured on the particle side and computed on
the field side.

**Part c)** Animate the cloud: $150$ frames of the walker histogram
spreading, with the Gaussian envelope of variance $2Dt$ riding on top.
The animation's physics is certified by the moment checks of Part b),
computed from the same ensemble the frames draw.

```{admonition} With your assistant
:class: tip
The FTCS update is three lines; the plumbing around it (grids,
boundary handling, snapshot bookkeeping) is boilerplate an assistant
writes reliably from this description. The check is yours, and it is
the stability boundary itself: rerun the solver at $r = 0.6$ and watch
the sawtooth instability erupt, then explain why the walk — which has
$r = 1/2$ exactly — sits precisely on the edge. The criterion, not the
code, is the physics.
```

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    max_diff < 2e-3,
    "the walker histogram and the FTCS solution of the diffusion "
    "equation agree pointwise at the noise floor: one object, two "
    "magnifications",
    f"max diff {max_diff:.1e} vs peak {pair_field.max():.1e}",
)
validate.check(
    all(
        abs(x2_checkpoints[t_c] - t_c) < 4.0 * np.sqrt(2.0 * t_c**2 / N_WALKERS)
        for t_c in (100, 200, 400)
    ),
    "the walkers' spread obeys <x²> = 2Dt = t at every checkpoint " "within 4σ",
    f"measured {[round(v, 1) for v in x2_checkpoints.values()]}",
)
validate.close(
    m2_field,
    400.0,
    "and the field solution carries exactly the same second moment",
    rtol=1e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 3 — First passage: certain, yet on average never

When does the walker first reach $+1$? {eq}`eq-rw-firstpassage` makes
two claims that sound contradictory until the tail is understood:
passage is certain, and the expected wait is infinite.

**Part a)** Release $3\times10^5$ walkers and record each one's first-
passage time to $+1$, capped at $T_{\max} = 20\,000$ steps. Verify the
tail: a log–log `numpy.polyfit` of the binned first-passage density
over $10 < t < 3000$ gives slope $-3/2$ within $\pm 0.05$.

**Part b)** Certain: verify more than $98.5\%$ of walkers have passed
by the cap (the exact passed-fraction deficit decays as
$\sim\sqrt{2/(\pi T_{\max})}$, about $0.6\%$ here). On average never:
verify the *capped* mean first-passage time grows with the cap like
$\sqrt{T_{\max}}$ — compute the sample mean of $\min(T, T_{\rm cap})$
at caps $5000$ and $20\,000$ and verify their ratio lands within
$15\%$ of $\sqrt{20000/5000} = 2$: the mean is not converging to
anything, and that *is* the measurement.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    slope_fp,
    -1.5,
    "the first-passage density decays as t^(-3/2): the reflection "
    "principle's tail, fitted over two decades",
    rtol=0.0,
    atol=0.05,
)
validate.check(
    frac_passed > 0.985,
    "passage is certain: all but the √(2/πT) stragglers have crossed by " "the cap",
    f"{frac_passed:.4f} passed",
)
validate.close(
    cap_ratio,
    2.0,
    "yet the capped mean wait grows as √cap — quadrupling the cap "
    "doubles the mean, which therefore converges to nothing",
    rtol=0.15,
)

## Exercise 4 — The gambler's ruin, settled exactly

{eq}`eq-rw-ruin` is one of the oldest exact results in probability, and
its logic deserves to be stated because it is pure physics: the fair
walk is a *martingale* (its expectation never moves), so the expected
final position must equal the starting one, which fixes the two
absorption probabilities; a second martingale, $x^2 - t$, fixes the
mean duration the same way.

**Part a)** A gambler starts with $a = 10$ units and plays fair
unit-stake rounds against a bank of $b = 20$, stopping at bankruptcy
($-a$ from the start) or at breaking the bank ($+b$). Simulate
$4\times10^5$ games. Verify the ruin probability $b/(a+b) = 2/3$
within $4$ binomial standard errors, and the mean game length
$ab = 200$ within $2\%$ — exact constants, met by an ensemble.

**Part b)** The asymmetry of consequences. The overall mean hides two
very different games: reaching the *far* wall requires a long
excursion, so bank-breaking games last much longer than ruinous ones.
Verify the ordering quantitatively: the conditional mean duration of
winning games exceeds that of ruined games by at least $40\%$. Fair
odds do not mean symmetric experiences.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    abs(p_ruin - B_BANK / (A_STAKE + B_BANK)) < 4.0 * se_ruin,
    "the martingale answer holds: ruin probability b/(a+b) = 2/3 "
    "within 4σ of the ensemble",
    f"{p_ruin:.4f} vs {B_BANK / (A_STAKE + B_BANK):.4f}",
)
validate.close(
    mean_T,
    float(A_STAKE * B_BANK),
    "and the second martingale x² − t prices the mean game at exactly "
    "ab = 200 rounds",
    rtol=0.02,
)
validate.check(
    mean_T_win > 1.4 * mean_T_ruin,
    "conditional durations are strongly asymmetric: breaking the "
    "distant bank takes far longer than going broke",
    f"win {mean_T_win:.0f} vs ruin {mean_T_ruin:.0f} rounds",
)

## Exercise 5 — Pólya's theorem: the drunkard and the bird

Recurrence is the walk's deepest dimension-dependence, and
{eq}`eq-rw-polya` makes the three-dimensional case *exactly*
computable: no simulation is needed for the answer, only for the
confirmation.

**Part a)** Evaluate the Glasser–Zucker closed form `polya_p3_exact`
(four `scipy.special.gamma` calls) and verify it gives
$p_3 = 0.340537$ to `rtol=1e-6`: about one return in three.

**Part b)** Confirm by ensemble. Walk $10^5$ walkers on the simple
cubic lattice for $3000$ steps (each step one of the six axis moves,
from the seeded generator) and record whether each ever revisits the
origin. The measured fraction *undershoots* $p_3$ by the walkers that
would have returned only after the cap (a deficit of order
$1/\sqrt{T}$, under a percent here): verify the measured fraction
lands in the one-sided window $[p_3 - 0.012,\ p_3 + 0.004]$.

**Part c)** The other two dimensions. Verify recurrence looks like
recurrence: in one dimension more than $97\%$ of $2\times10^4$ walkers
return within $3000$ steps; in two dimensions the return fraction by
the same cap is far lower (logarithmically slow — the drunkard *does*
get home, but only just) yet clearly above the three-dimensional
plateau: verify the ordering $p_1^{(3000)} > 0.97 > p_2^{(3000)} >
p_3^{(3000)} + 0.25$. One theorem, three phenomenologies.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    p3_exact,
    0.340537,
    "the Glasser–Zucker Gamma-function formula prices the 3D return at "
    "0.340537: one homecoming in three, exactly",
    rtol=1e-6,
)
validate.check(
    p3_exact - 0.012 < p3_mc < p3_exact + 0.004,
    "the ensemble confirms it, undershooting only by the late returners "
    "beyond the cap",
    f"MC {p3_mc:.4f} vs exact {p3_exact:.4f}",
)
validate.check(
    p1_mc > 0.97 and 0.97 > p2_mc > p3_mc + 0.25,
    "and dimension decides the phenomenology: 1D returns almost surely "
    "within the cap, 2D climbs logarithmically, 3D has saturated",
    f"p1 {p1_mc:.3f}, p2 {p2_mc:.3f}, p3 {p3_mc:.3f}",
)

## Exercise 6 — The self-avoiding walk: a polymer swells

Forbid self-intersection and the walk becomes the minimal polymer.
The price is sampling: growing naively and discarding trapped walks
biases the survivors toward compact shapes, and the fix — Rosenbluth
weighting, growing among *open* neighbours while accumulating the
open-neighbour-count product as a compensating weight — is the
founding importance-sampling algorithm of polymer physics.
**Write this one yourself** — the implementation is the lesson.

**Part a)** With `rosenbluth_r2`, estimate $\langle R^2\rangle$ for
two-dimensional self-avoiding walks of $N = 20, 40, 80$ steps
($20\,000$ attempted walks each: the Rosenbluth weights are
heavy-tailed, so the estimator earns its precision slowly). Fit $\log\langle R^2\rangle$ against $\log N$
(`numpy.polyfit`, degree 1) and verify the exponent $\nu$ (half the
slope) lands in $[0.70, 0.80]$, containing the exact
two-dimensional $\nu = 3/4$ — and decisively above the simple walk's
$1/2$.

**Part b)** The control experiment: the same fit for the *simple* walk
(whose $\langle R^2\rangle = N$ exactly in 2D as well) must give
$\nu = 1/2$ within $\pm 0.02$. Excluded volume does not correct the
exponent; it replaces it — the same lesson
[§5.10](ising-emergence-universality.ipynb) taught with critical
exponents, arriving here on foot.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    0.70 < nu_saw < 0.80,
    "the Rosenbluth-sampled self-avoiding walk swells with the exact 2D "
    "Flory exponent ν = 3/4 inside the fit window",
    f"ν = {nu_saw:.3f}",
)
validate.close(
    nu_simple,
    0.5,
    "while the simple-walk control fits ν = 1/2 exactly: the exponent "
    "change is excluded volume, not statistics",
    rtol=0.0,
    atol=0.02,
)

## Notebook summary

- The simple walk met its exact binomial bin by bin (total variation
  $4\times10^{-3}$, worst bin under $3\sigma$), obeyed
  $\langle x^2\rangle = N$ within $4\sigma$, and marched monotonically
  onto the Gaussian (deviation $<2\%$ by $N = 256$): the $\sqrt N$ law
  and the CLT, both watched.
- The walker histogram *solved the diffusion equation*: parity-summed
  against the FTCS solution at $D = a^2/2\tau = 1/2$, the two agreed at
  the counting-noise floor, with second moments matching $2Dt$ on both
  the particle and field sides.
- First passage delivered its paradox quantitatively: slope $-1.50$
  over two decades, $98.9\%$ passed by the cap, and a capped mean that
  doubles when the cap quadruples — certain, yet on average never.
- The martingale arguments held to Monte Carlo precision: ruin at
  $b/(a+b) = 2/3$, mean game exactly $ab = 200$ rounds, with strongly
  asymmetric conditional durations.
- Pólya's theorem arrived twice: exactly ($p_3 = 0.340537$ from four
  Gamma functions) and empirically (the ensemble undershooting by its
  late returners), with the 1D/2D/3D ordering displaying all three
  phenomenologies.
- The self-avoiding walk swelled with $\nu \approx 3/4$ against the
  simple walk's control $\nu = 1/2$: excluded volume changes the
  universality class, measured with the Rosenbluth weights that founded
  polymer Monte Carlo.

## Outlook

- **Back to Langevin.** [§5.11](taste-of-nonequilibrium.ipynb)'s
  Brownian particle is this notebook's walk with inertia and a real
  thermostat: the Einstein relation $D = k_BT/\gamma$ prices our
  lattice $D$ in physical units, and the Green–Kubo integral is the
  continuum limit of summing step correlations.
- **First passage everywhere.** The $t^{-3/2}$ law reappears in
  reaction kinetics (diffusion-limited encounter times), in neuronal
  integrate-and-fire models, and in the pricing of barrier options;
  the gambler's-ruin martingale argument is the same one that prices
  them.
- **Polymers, properly.** Rosenbluth weighting grew into the pruned-
  enriched methods (PERM) that simulate thousand-monomer chains, and
  the $\nu$ exponent's exact 2D value comes from conformal field
  theory — the same web of universality that
  [§5.10](ising-emergence-universality.ipynb) opened.
- **Quantum walks.** Replace the coin by a unitary and interference
  makes the walker spread *linearly* in time — the quadratic speedup
  behind several quantum algorithms, and a bridge back to
  [§6.27](../06-quantum-mechanics/quantum-information.ipynb).

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()